In [3]:
%pip install scrapy selenium scrapy-selenium webdriver-manager

Note: you may need to restart the kernel to use updated packages.


In [4]:
from scrapy.utils.project import get_project_settings
from scrapy_selenium import SeleniumMiddleware

settings = get_project_settings()
settings.set('SELENIUM_DRIVER_NAME', 'chrome')
settings.set('SELENIUM_DRIVER_EXECUTABLE_PATH', '/usr/local/bin/chromedriver')
settings.set('SELENIUM_DRIVER_ARGUMENTS', ['--headless', '--disable-gpu', '--no-sandbox'])
settings.set('DOWNLOADER_MIDDLEWARES', {
    'scrapy_selenium.SeleniumMiddleware': 800,
})


In [9]:
import scrapy
from scrapy_selenium import SeleniumRequest

class TripAdvisorSpider(scrapy.Spider):
    name = 'tripadvisor_spider'
    allowed_domains = ['tripadvisor.com']
    start_urls = [
        'https://www.tripadvisor.com/Airline_Review-d8729060-Reviews-Delta-Air-Lines'
    ]

    def start_requests(self):
        for url in self.start_urls:
            yield SeleniumRequest(url=url, callback=self.parse)

    def parse(self, response):
        # Extract hotel listings
        hotels = response.xpath('//div[contains(@class, "listing_title")]/a')
        for hotel in hotels:
            yield {
                'name': hotel.xpath('text()').get(),
                'url': hotel.xpath('@href').get()
            }

from scrapy.crawler import CrawlerProcess
process = CrawlerProcess()
process.crawl(TripAdvisorSpider)
process.start()

2025-03-17 19:44:23 [scrapy.utils.log] INFO: Scrapy 2.11.1 started (bot: scrapybot)
2025-03-17 19:44:23 [scrapy.utils.log] INFO: Versions: lxml 5.2.1.0, libxml2 2.13.1, cssselect 1.2.0, parsel 1.8.1, w3lib 2.1.2, Twisted 23.10.0, Python 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:22:19) [Clang 14.0.6 ], pyOpenSSL 24.2.1 (OpenSSL 3.0.15 3 Sep 2024), cryptography 43.0.0, Platform macOS-14.1.2-arm64-arm-64bit
2025-03-17 19:44:23 [scrapy.addons] INFO: Enabled addons:
[]
2025-03-17 19:44:23 [py.warnings] WARNING: /opt/homebrew/anaconda3/lib/python3.12/site-packages/scrapy/utils/request.py:254: ScrapyDeprecationWarning: '2.6' is a deprecated value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting.

It is also the default value. In other words, it is normal to get this warning if you have not defined a value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting. This is so for backward compatibility reasons, but it will change in a future version of Scrapy.

See the doc

ReactorNotRestartable: 

In [10]:
import scrapy
from scrapy.crawler import CrawlerProcess
from scrapy.selector import Selector
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager

class TripAdvisorSpider(scrapy.Spider):
    name = 'tripadvisor_spider'
    allowed_domains = ['tripadvisor.com']
    start_urls = [
        'https://www.tripadvisor.com/Hotels-g28953-New_York-Hotels.html'
    ]

    def __init__(self):
        self.driver = webdriver.Chrome(ChromeDriverManager().install())
        super().__init__()

    def start_requests(self):
        for url in self.start_urls:
            yield scrapy.Request(url=url, callback=self.parse)

    def parse(self, response):
        self.driver.get(response.url)
        # Add your parsing logic here

    def closed(self, reason):
        self.driver.quit()

# Set up the CrawlerProcess
process = CrawlerProcess()
process.crawl(TripAdvisorSpider)
process.start()


2025-03-17 19:45:09 [scrapy.utils.log] INFO: Scrapy 2.11.1 started (bot: scrapybot)
2025-03-17 19:45:09 [scrapy.utils.log] INFO: Versions: lxml 5.2.1.0, libxml2 2.13.1, cssselect 1.2.0, parsel 1.8.1, w3lib 2.1.2, Twisted 23.10.0, Python 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:22:19) [Clang 14.0.6 ], pyOpenSSL 24.2.1 (OpenSSL 3.0.15 3 Sep 2024), cryptography 43.0.0, Platform macOS-14.1.2-arm64-arm-64bit
2025-03-17 19:45:09 [WDM] INFO: ====== WebDriver manager ======
2025-03-17 19:45:09 [WDM] INFO: Get LATEST chromedriver version for google-chrome
2025-03-17 19:45:09 [urllib3.connectionpool] DEBUG: Starting new HTTPS connection (1): googlechromelabs.github.io:443
2025-03-17 19:45:09 [urllib3.connectionpool] DEBUG: https://googlechromelabs.github.io:443 "GET /chrome-for-testing/latest-patch-versions-per-build.json HTTP/11" 200 10195
2025-03-17 19:45:09 [WDM] INFO: Get LATEST chromedriver version for google-chrome
2025-03-17 19:45:09 [urllib3.connectionpool] DEBUG: Sta